# Synthetic vector-borne disease model

Humans (SEIRS, four age bands) and mosquitoes (SEI) coupled by Ross-Macdonald biting. No real
data: every parameter is illustrative. The model is built in `vbd_model.model`, whose module
docstring gives the formulas.

- **Seasonality**: mosquito emergence follows a cosine over the year
  (`seasonal_amplitude`, peaking on `peak_emergence_day`), so mosquito density, and with it
  transmission, cycles annually.
- **Age heterogeneity in bite rate**: `rel_bite_<age>` is each band's relative attractiveness
  to mosquitoes. Bites are shared out in proportion to `rel_bite × population`, so it affects
  both who gets infected and who infects mosquitoes.
- **Age heterogeneity in susceptibility**: `rel_susc_<age>` scales the chance an infectious
  bite infects.
- **Intervention**: `bite_reduction` cuts bites per mosquito (bednets, repellents) from
  `intervention_start` onwards.

In [ ]:
import numpy as np
import pandas as pd

pd.options.plotting.backend = "plotly"

from summer4 import Compartments, Epoch, FlowMass, OutputSet

from vbd_model.model import DEFAULT_PARAMS, HUMANS, MOSQUITOES_PER_HUMAN, age, build_model, hstate, mstate, r0, species

In [ ]:
params = dict(DEFAULT_PARAMS)
pd.Series(params, name="value").to_frame()

In [ ]:
model = build_model()
compiled = model.compile()
compiled.pmap.labels()

R0 from the next-generation matrix (`r0`), at the mean mosquito density (season-averaged), with
and without the age heterogeneity:

In [ ]:
flat = {**params, **{f"rel_bite_{a}": 1.0 for a in age.traits}, **{f"rel_susc_{a}": 1.0 for a in age.traits}}
pd.Series({"with age heterogeneity": r0(params), "no age effects (all multipliers 1)": r0(flat)}, name="R0")

## Baseline: ten years, no intervention

In [ ]:
outputs = OutputSet()
outputs["incidence"] = FlowMass("human_infection").sum_by()
outputs["incidence_by_age"] = FlowMass("human_infection").sum_by(age, side="source")
outputs["population_by_age"] = Compartments(where=species["human"]).sum_by(age)
outputs["mosquitoes_per_human"] = Compartments(where=species["mosquito"]).sum_by() / HUMANS
outputs["human_prevalence"] = Compartments(where=hstate["I"]).sum_by() / HUMANS
outputs["mosquito_prevalence"] = Compartments(where=mstate["I"]).sum_by() / Compartments(where=species["mosquito"]).sum_by()
outputs["immune_by_age"] = Compartments(where=hstate["R"]).sum_by(age) / outputs.ref("population_by_age")

epoch = Epoch(pd.Timestamp("2025-01-01"))
run_kwargs = dict(t0=0.0, t1=10 * 365.0, dt=1.0, epoch=epoch, solver="tsit5")
named = outputs.evaluate(compiled.run(params, save=outputs.plan(), **run_kwargs), params)
series = lambda key: named[key].to_pandas()["value"]

In [ ]:
pd.concat(
    [series("mosquitoes_per_human").rename("mosquitoes per human"),
     (100 * series("mosquito_prevalence")).rename("% mosquitoes infectious"),
     (100 * series("human_prevalence")).rename("% humans infectious")],
    axis=1,
).plot(title="Seasonal mosquito density drives annual epidemics", labels={"value": "", "time": ""})

In [ ]:
weekly = 1000 * named["incidence"].integrate_intervals().resample("W", how="sum").to_pandas()["value"] / HUMANS
weekly.plot(title="Weekly infections per 1,000 people", labels={"value": "per 1,000", "time": ""})

In [ ]:
by_month = weekly.loc["2030":].groupby(weekly.loc["2030":].index.month).mean()
by_month.index.name = "month"
by_month.round(2).to_frame("mean weekly infections per 1,000 (2030-34)").T

Mosquito density swings between 1.1 and 8.9 per human, peaking around 23 February, about nine days after the emergence peak (day 45). Transmission builds over the following two months: human infectious prevalence peaks at about 6% in late April, and infected mosquitoes peak in mid-May. From 2026 the epidemic repeats every year at about 670 infections per 1,000 people (immunity lasts a year, so most people are infected roughly once a year). Weekly incidence averages 61 per 1,000 in April and under 0.4 per 1,000 from September to December. The deterministic model's trough is about 15 infectious people, so in a real population, or a stochastic version of this model, the infection could die out between seasons.

Annual infections per 1,000 people, by age band:

In [ ]:
annual = named["incidence_by_age"].integrate_intervals().resample("YE", how="sum").to_pandas()
pop = named["population_by_age"].to_pandas().iloc[0]
annual_rate = 1000 * annual / pop.values
annual_rate.index = annual_rate.index.year
annual_rate.round(0)

In [ ]:
named["immune_by_age"].to_pandas().plot(title="Proportion immune (R), by age",
                                        labels={"value": "proportion", "time": ""})

With the defaults the two age gradients largely cancel. Children are bitten least (`rel_bite_0 = 0.5`) but are the most susceptible (`rel_susc_0 = 1.0`), so the product `rel_bite × rel_susc` is 0.50, 0.72, 0.70 and 0.66 across the bands. Annual incidence is nearly flat by age, at 698 per 1,000 in under-5s and 663 in the 40+ band; under-5s are slightly higher because they are born susceptible. The heterogeneity still matters for transmission: bites concentrated on less susceptible adults lower R0 from 2.37 to 1.97.

## Scenarios: reducing bites from 2030

Bite reductions of 10%, 20% and 30% starting on 1 January 2030 (day 1825), run to 2039. Only
parameters change, so this reuses the compiled model.

In [ ]:
reductions = [0.0, 0.1, 0.2, 0.3]
long_kwargs = {**run_kwargs, "t1": 15 * 365.0}
scenario_weekly, scenario_annual, scenario_r0 = {}, {}, {}
for red in reductions:
    p = {**params, "bite_reduction": red}
    res = outputs.evaluate(compiled.run(p, save=outputs.plan(), **long_kwargs), p)
    counts = res["incidence"].integrate_intervals()
    label = f"{red:.0%} reduction"
    scenario_weekly[label] = 1000 * counts.resample("W", how="sum").to_pandas()["value"] / HUMANS
    scenario_annual[label] = 1000 * counts.resample("YE", how="sum").to_pandas()["value"] / HUMANS
    scenario_r0[label] = r0(p)

pd.Series(scenario_r0, name="R0 once the intervention is in place").round(2).to_frame()

In [ ]:
pd.DataFrame(scenario_weekly).loc["2028":].plot(
    title="Weekly infections per 1,000 people, by bite reduction from 2030",
    labels={"value": "per 1,000", "time": "", "variable": ""},
)

In [ ]:
annual_scenarios = pd.DataFrame(scenario_annual)
annual_scenarios.index = annual_scenarios.index.year
annual_scenarios.round(0)

All three reductions leave R0 above 1 (1.77, 1.57 and 1.38), so none eliminates infection, but they change the dynamics in different ways. A 10% reduction cuts annual infections only slightly, from 671 to between 601 and 626 per 1,000, with damped alternation between years. A 20% reduction pushes the system into a two-year cycle: by the late 2030s years of about 150 alternate with years of about 830 infections per 1,000, so some years are worse than without the intervention. A 30% reduction gives a long lull (111 per 1,000 in 2030), then settles to a smaller annual epidemic of about 380 per 1,000 (a 43% drop), peaking in early June rather than late April. These are deterministic results; with small numbers in the troughs, a stochastic model would probably show fade-outs and reintroductions.